# Data Processing for NLP
## Load, clean, and preprocess data for NLP tasks

In [ ]:
import pandas as pd
import numpy as np
import os
import csv
from typing import Tuple, List, Dict, Optional
from sklearn.model_selection import train_test_split

## DataProcessor Class

In [ ]:
class DataProcessor:
    """Handles data loading and preprocessing"""
    
    @staticmethod
    def load_csv(filepath: str, encoding: str = 'latin-1') -> pd.DataFrame:
        """Load CSV file with error handling"""
        try:
            df = pd.read_csv(filepath, encoding=encoding)
            print(f"✓ Loaded {filepath}: {df.shape[0]} rows, {df.shape[1]} columns")
            return df
        except Exception as e:
            print(f"✗ Error loading {filepath}: {e}")
            return None
    
    @staticmethod
    def validate_data(df: pd.DataFrame, text_col: str, label_col: str) -> bool:
        """Validate data structure and content"""
        if text_col not in df.columns:
            print(f"✗ Text column '{text_col}' not found")
            return False
        if label_col not in df.columns:
            print(f"✗ Label column '{label_col}' not found")
            return False
        
        missing_text = df[text_col].isnull().sum()
        missing_label = df[label_col].isnull().sum()
        
        if missing_text > 0:
            print(f"⚠ Warning: {missing_text} missing values in '{text_col}'")
        if missing_label > 0:
            print(f"⚠ Warning: {missing_label} missing values in '{label_col}'")
        
        print(f"✓ Data validation passed")
        return True
    
    @staticmethod
    def get_data_statistics(df: pd.DataFrame, text_col: str, label_col: str) -> Dict:
        """Get statistics about the data"""
        stats = {
            'total_rows': len(df),
            'total_columns': len(df.columns),
            'text_column': text_col,
            'label_column': label_col,
            'unique_labels': df[label_col].nunique(),
            'label_distribution': df[label_col].value_counts().to_dict(),
            'avg_text_length': df[text_col].apply(lambda x: len(str(x).split())).mean(),
            'max_text_length': df[text_col].apply(lambda x: len(str(x).split())).max(),
            'min_text_length': df[text_col].apply(lambda x: len(str(x).split())).min(),
            'missing_text': df[text_col].isnull().sum(),
            'missing_label': df[label_col].isnull().sum(),
        }
        return stats

## Sample Data Generator

In [ ]:
class SampleDataGenerator:
    """Generate sample data for testing and demonstration"""
    
    @staticmethod
    def generate_cybersecurity_samples(n_samples: int = 100) -> pd.DataFrame:
        """Generate sample cybersecurity attack descriptions"""
        
        attacks = {
            'SQL_Injection': [
                "Attacker injected malicious SQL code into login field",
                "SQL injection detected in search parameter",
                "Database query manipulation attempt captured",
                "Suspicious SQL syntax in user input detected",
                "SQL command injection in form submission"
            ],
            'XSS': [
                "Cross-site scripting payload detected in comments",
                "JavaScript code injected into web form",
                "XSS attack attempt in user profile section",
                "Malicious script tag found in user input",
                "DOM-based XSS vulnerability exploited"
            ],
            'DDoS': [
                "Massive traffic spike detected from single IP",
                "Distributed denial of service attack in progress",
                "Bot network flooding server with requests",
                "Unusual traffic patterns indicating DDoS attack",
                "Multiple simultaneous connection attempts blocked"
            ],
            'Normal': [
                "Regular user login from expected location",
                "Routine database backup completed successfully",
                "Normal web traffic patterns observed",
                "Standard file access by authenticated user",
                "Regular system maintenance activity logged"
            ]
        }
        
        data = []
        np.random.seed(42)
        
        for _ in range(n_samples):
            attack_type = np.random.choice(list(attacks.keys()))
            description = np.random.choice(attacks[attack_type])
            data.append({
                'attack_type': attack_type,
                'description': description,
                'timestamp': pd.Timestamp.now() - pd.Timedelta(days=np.random.randint(0, 365)),
                'severity': np.random.choice(['Low', 'Medium', 'High', 'Critical'])
            })
        
        df = pd.DataFrame(data)
        print(f"✓ Generated {n_samples} cybersecurity samples")
        return df
    
    @staticmethod
    def save_samples(df: pd.DataFrame, filepath: str) -> None:
        """Save sample data to CSV"""
        df.to_csv(filepath, index=False, encoding='utf-8')
        print(f"✓ Saved {len(df)} samples to {filepath}")

## Generate and Process Sample Data

In [ ]:
# Generate cybersecurity samples
print("Generating cybersecurity samples...")
cyber_samples = SampleDataGenerator.generate_cybersecurity_samples(n_samples=200)
SampleDataGenerator.save_samples(cyber_samples, './sample_cybersecurity.csv')
print(f"\nFirst few rows:")
cyber_samples.head()

## Data Statistics and Analysis

In [ ]:
# Get data statistics
processor = DataProcessor()
stats = processor.get_data_statistics(cyber_samples, 'description', 'attack_type')

print("\n" + "="*50)
print("DATA STATISTICS")
print("="*50)
print(f"Total rows: {stats['total_rows']}")
print(f"Total columns: {stats['total_columns']}")
print(f"Unique labels: {stats['unique_labels']}")
print(f"Missing text values: {stats['missing_text']}")
print(f"Missing label values: {stats['missing_label']}")
print(f"\nText length statistics:")
print(f"  Average: {stats['avg_text_length']:.1f} words")
print(f"  Max: {stats['max_text_length']:.0f} words")
print(f"  Min: {stats['min_text_length']:.0f} words")
print(f"\nLabel distribution:")
for label, count in stats['label_distribution'].items():
    percentage = (count / stats['total_rows']) * 100
    print(f"  {label}: {count} ({percentage:.1f}%)")

## Data Split

In [ ]:
def split_data(df, test_size=0.2, val_size=0.1):
    """Split data into train, validation, and test sets"""
    temp_size = test_size + val_size
    train_df, temp_df = train_test_split(df, test_size=temp_size, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=test_size / temp_size, random_state=42)
    
    print(f"✓ Data split:")
    print(f"  Train: {len(train_df)} rows ({len(train_df)/len(df)*100:.1f}%)")
    print(f"  Val:   {len(val_df)} rows ({len(val_df)/len(df)*100:.1f}%)")
    print(f"  Test:  {len(test_df)} rows ({len(test_df)/len(df)*100:.1f}%)")
    
    return train_df, val_df, test_df

# Split data
train_df, val_df, test_df = split_data(cyber_samples, test_size=0.2, val_size=0.1)